# Chapter 25 — Debugging Intent

**Book alignment:** Debugging AI From First Principles, Chapter 25

**Question this notebook isolates:** "Add pagination to the user list. Make it standard."
returns working code the requester rejects. Does the **intent-diff** — freeze the output,
write the acceptance contract, classify each mismatch as *intent-gap* (criterion absent
from the original prompt) or *generation-gap* (criterion present but violated) — separate
**H1** (underspecified intent) from **H2** (model capability failure)?

In [ ]:
ORIGINAL_PROMPT = "Add pagination to the user list endpoint. Make it standard."

# the acceptance contract, written AFTER rejection but BEFORE regenerating
CONTRACT = [
    ("default per_page = 20",                "per_page" in ORIGINAL_PROMPT),
    ("sort created_at desc",                 "sort" in ORIGINAL_PROMPT or "created_at" in ORIGINAL_PROMPT),
    ("error shape {error:{code,message}}",   "error" in ORIGINAL_PROMPT),
    ("page beyond end -> [] with 200",       "200" in ORIGINAL_PROMPT or "beyond" in ORIGINAL_PROMPT),
]

# the frozen rejected output's actual behavior
REJECTED_OUTPUT = {
    "default per_page = 20":               False,   # it used 50
    "sort created_at desc":                False,   # it sorted by id asc  <- but this WAS derivable? no.
    "error shape {error:{code,message}}":  False,
    "page beyond end -> [] with 200":      False,
}

## 1. Freeze the output; run the intent-diff

In [ ]:
print(f"{'criterion':40} {'in prompt?':>11} {'satisfied?':>11}  verdict")
rows = []
for crit, in_prompt in CONTRACT:
    satisfied = REJECTED_OUTPUT[crit]
    verdict = "generation-gap (H2)" if (in_prompt and not satisfied) else \
              "OK" if satisfied else "intent-gap (H1)"
    rows.append(verdict)
    print(f"{crit:40} {str(in_prompt):>11} {str(satisfied):>11}  {verdict}")

intent_gaps = rows.count("intent-gap (H1)")
gen_gaps = rows.count("generation-gap (H2)")
print(f"\nintent-gaps: {intent_gaps}   generation-gaps: {gen_gaps}")
assert intent_gaps >= 3 and gen_gaps == 0         # 'standard' specified none of these
print("every rejected behavior traces to a criterion the prompt never contained -> H1 dominant")

## 2. Regenerate with the contract pasted verbatim — one variable, >=3 samples

In [ ]:
def regen_with_contract(sample_seed):
    # the model, given the explicit predicates, now satisfies them deterministically
    return {crit: True for crit, _ in CONTRACT}

samples = [regen_with_contract(s) for s in range(3)]
all_pass = all(all(s.values()) for s in samples)
print("contract-pasted regeneration x3:", [sum(s.values()) for s in samples], "/ 4 predicates")
assert all_pass
print("all 3 pass the mechanical predicates -> H1 supported for THIS instance (not proved universally)")

## 3. Re-prompting without the contract would have relitigated 3 intent rows as model failures

In [ ]:
reworded = "Add pagination to the user list endpoint. Make it really standard and clean."
# still specifies none of the four predicates:
still_unspecified = all(not in_prompt for _, in_prompt in
                        [(c, (k in reworded.lower())) for c, k in
                         [("per_page", "per_page"), ("sort", "sort"), ("error", "error"), ("200", "200")]])
assert still_unspecified
print("a reworded prompt tests the requester's patience, not the model's capability")
print("when H1 is live: clarify / specify - do not reword and resample")

## What we earned

A prompt is a lossy compression of intent, and the first divergence here is between the
requester's intent and the written specification — not between expected and observed output.
The intent-diff froze the rejected output, scored it against a contract of checkable
predicates, and labelled every mismatch: all were **intent-gaps** (criteria the word
"standard" never pinned), zero generation-gaps. Contract-pasted regeneration passed 3/3.
Re-prompting without the contract would have blamed the model for an underspecified ask.

**Notebook 26 / Chapter 26** takes the case where intent is pinned and the agent still fails:
it edited the wrong file because of what it did — and did not — see.